In [ ]:
# Loss-landscape mockup for Gradient Consistency
# Works in a Jupyter notebook. Produces:
#  - Matplotlib 3D surfaces (static, publication-friendly)
#  - Optional Plotly interactive 3D surfaces (nice for exploring camera angles)
#
# Surface definition (per your paper):
# ΔL̃(u1, dL/du2; dL/dS) = (dL/dS) * ΔS(u1) + (dL/du2) * Δu2(u1)
# where:
#   ΔS = +1 if u1 < Vth else -1
#   Δu2 = -u1 if u1 < Vth else Vth

import numpy as np

# -----------------------------
# Core surface definition
# -----------------------------
def deltaS(u1, Vth=1.0):
    return np.where(u1 < Vth, 1.0, -1.0)

def delta_u2(u1, Vth=1.0):
    return np.where(u1 < Vth, -u1, Vth)

def delta_L_tilde(U1, dL_du2, dL_dS=1.0, Vth=1.0):
    """
    Vectorized. U1 and dL_du2 can be scalars or arrays broadcastable to same shape.
    """
    return dL_dS * deltaS(U1, Vth) + dL_du2 * delta_u2(U1, Vth)

# -----------------------------
# Grid helpers
# -----------------------------
def make_grid(
    u1_min=0.0, u1_max=2.0, u1_n=250,
    dLdu2_min=-2.0, dLdu2_max=2.0, dLdu2_n=250
):
    u1 = np.linspace(u1_min, u1_max, u1_n)
    dLdu2 = np.linspace(dLdu2_min, dLdu2_max, dLdu2_n)
    U1, D = np.meshgrid(u1, dLdu2, indexing="xy")
    return U1, D

# -----------------------------
# Matplotlib (static)
# -----------------------------
def plot_matplotlib_surfaces(
    dL_dS_values=(-1.0, 0.0, 1.0),
    Vth=1.0,
    u1_range=(0.0, 2.0),
    dLdu2_range=(-2.0, 2.0),
    grid_n=500,
    z_clip=None,  # e.g. (-4, 4) to clip z-range for readability
    elev=25,
    azim=-60,
    add_threshold_plane=True,
):
    import matplotlib.pyplot as plt
    from matplotlib.colors import TwoSlopeNorm

    U1, D = make_grid(
        u1_min=u1_range[0], u1_max=u1_range[1], u1_n=grid_n,
        dLdu2_min=dLdu2_range[0], dLdu2_max=dLdu2_range[1], dLdu2_n=grid_n
    )

    n = len(dL_dS_values)
    fig = plt.figure(figsize=(6.5 * n, 5.5))

    # Common normalization: center at 0 to emphasize consistent vs inconsistent sign
    # (ΔL̃ < 0 consistent; ΔL̃ > 0 inconsistent)
    # We'll compute per-panel z-lims unless z_clip is set.
    for i, dL_dS in enumerate(dL_dS_values, start=1):
        ax = fig.add_subplot(1, n, i, projection="3d")

        Z = delta_L_tilde(U1, D, dL_dS=dL_dS, Vth=Vth)

        if z_clip is not None:
            Z_plot = np.clip(Z, z_clip[0], z_clip[1])
            zmin, zmax = z_clip
        else:
            Z_plot = Z
            zmin, zmax = np.min(Z_plot), np.max(Z_plot)

        # Diverging colormap centered at 0 (do not specify exact colors; use default)
        norm = TwoSlopeNorm(vmin=zmin, vcenter=0.0, vmax=zmax)

        surf = ax.plot_surface(
            U1, D, Z_plot,
            rstride=2, cstride=2,
            linewidth=0, antialiased=True,
            norm=norm
        )

        ax.set_title(rf"$\partial L/\partial S = {dL_dS:g}$", pad=12)
        ax.set_xlabel(r"$u_1$")
        ax.set_ylabel(r"$\partial L/\partial u_2$")
        ax.set_zlabel(r"$\Delta \tilde{L}$")

        ax.view_init(elev=elev, azim=azim)

        # Optional: show the threshold plane u1 = Vth as a vertical translucent sheet
        if add_threshold_plane:
            # Build a thin plane at u1=Vth spanning y-range and z-range
            yy = np.linspace(dLdu2_range[0], dLdu2_range[1], 50)
            zz = np.linspace(zmin, zmax, 50)
            YY, ZZ = np.meshgrid(yy, zz, indexing="xy")
            XX = np.full_like(YY, Vth)
            ax.plot_surface(XX, YY, ZZ, alpha=0.15, linewidth=0)

        # Mark consistency boundary ΔL̃=0 via contour projected to bottom (optional)
        # This can be useful in print. Comment out if it clutters.
        try:
            ax.contour(
                U1, D, Z,
                levels=[0.0],
                offset=zmin,
                linewidths=2
            )
        except Exception:
            pass

        # Add a colorbar per axis
        fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.08, aspect=18)

        # Aesthetics
        ax.set_xlim(u1_range)
        ax.set_ylim(dLdu2_range)
        ax.set_zlim((zmin, zmax))

    plt.tight_layout()
    plt.show()

# -----------------------------
# Plotly (interactive)
# -----------------------------
def plot_plotly_surfaces(
    dL_dS_values=(-1.0, 0.0, 1.0),
    Vth=1.0,
    u1_range=(0.0, 2.0),
    dLdu2_range=(-2.0, 2.0),
    grid_n=200,
    z_clip=None,
    add_threshold_plane=True,
):
    """
    Requires: pip install plotly
    In Jupyter, this will render interactively (rotate/zoom).
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    U1, D = make_grid(
        u1_min=u1_range[0], u1_max=u1_range[1], u1_n=grid_n,
        dLdu2_min=dLdu2_range[0], dLdu2_max=dLdu2_range[1], dLdu2_n=grid_n
    )

    n = len(dL_dS_values)
    fig = make_subplots(
        rows=1, cols=n,
        specs=[[{"type": "surface"}] * n],
        subplot_titles=[rf"dL/dS = {v:g}" for v in dL_dS_values],
        horizontal_spacing=0.03
    )

    # We'll use a diverging colorscale centered around zero in display logic
    # by setting zmin/zmax symmetric if not clipped.
    for j, dL_dS in enumerate(dL_dS_values, start=1):
        Z = delta_L_tilde(U1, D, dL_dS=dL_dS, Vth=Vth)

        if z_clip is not None:
            Zp = np.clip(Z, z_clip[0], z_clip[1])
            zmin, zmax = z_clip
        else:
            # Symmetric bounds around 0 so sign is visually stable across panels
            m = np.max(np.abs(Z))
            zmin, zmax = -m, m
            Zp = Z

        fig.add_trace(
            go.Surface(
                x=U1, y=D, z=Zp,
                cmin=zmin, cmax=zmax,
                showscale=(j == n),  # one colorbar on the last panel
                opacity=1.0
            ),
            row=1, col=j
        )

        if add_threshold_plane:
            # Threshold plane at x=Vth with low opacity
            yy = np.linspace(dLdu2_range[0], dLdu2_range[1], 60)
            zz = np.linspace(zmin, zmax, 60)
            YY, ZZ = np.meshgrid(yy, zz, indexing="xy")
            XX = np.full_like(YY, Vth)

            fig.add_trace(
                go.Surface(
                    x=XX, y=YY, z=ZZ,
                    opacity=0.15,
                    showscale=False
                ),
                row=1, col=j
            )

    # Layout: per-subplot axis labels are limited; we set global scene styling
    # Plotly uses scene, scene2, ... naming
    for j in range(1, n + 1):
        scene_name = "scene" if j == 1 else f"scene{j}"
        fig.update_layout(**{
            scene_name: dict(
                xaxis_title="u1",
                yaxis_title="dL/du2",
                zaxis_title="ΔL~",
                xaxis=dict(range=list(u1_range)),
                yaxis=dict(range=list(dLdu2_range)),
                zaxis=dict(range=[zmin, zmax]),
            )
        })

    fig.update_layout(
        title="Effective jump-induced loss change surface: ΔL̃(u1, dL/du2; dL/dS)",
        height=550,
        width=520 * n,
        margin=dict(l=0, r=0, t=45, b=0),
    )
    fig.show()

# -----------------------------
# Example usage
# -----------------------------
# 1) Matplotlib: good for paper figures
plot_matplotlib_surfaces(
    dL_dS_values=(-1.0, 0.0, 1.0),
    Vth=1.0,
    u1_range=(0.0, 2.0),
    dLdu2_range=(-2.0, 2.0),
    grid_n=250,
    z_clip=None,         # try (-4, 4) if you want clipping
    elev=25,
    azim=120,
    add_threshold_plane=False
)

# 2) Plotly: interactive exploration (uncomment if you have plotly installed)
# plot_plotly_surfaces(
#     dL_dS_values=(-1.0, 0.0, 1.0),
#     Vth=1.0,
#     u1_range=(0.0, 2.0),
#     dLdu2_range=(-2.0, 2.0),
#     grid_n=200,
#     z_clip=None,
#     add_threshold_plane=True
# )
